**Project Objective** : Develop a custom chat agent that utilizes pre-trained ML Models. The chat agent offers mortality prediction and survival analysis and recommends medication to lower mortality risk for female patients admitted to ICU diagnosed with blood diseases (ICD-9: 280–289).

=================================================================

This Colab file

1. Installs Libraries that are needed for chat agent implementation
2. Loads Dataset and pre-trained model files. The model files are large and loaded from Google Drive Storage.Expectation is for the user to download the files from the shared location, create the suggested folder structure in SECTION 3, upload the files.
3. Implements Custom Action for Mortality Prediction by using a pretrained model
4. Implements Custom Action for Medication Recommendation using a pretrained model.
5. Acts as an ICU Doctor-Persona Chat Agent
6. Implements Test Case 1: Uses Real Patient data (based of MIMICIII Dataset) from csv file
7. Implements Test Case 2: Uses Real Patient data (based of MIMICIII Dataset) from csv file
8. Clinical Safety Notes and disclaimers.

Note: This is a Chat agent only and all advice is based on MIMICIII dataset data and pretrained model.The key guiding factor is that chat agent should not directly “guess” medications. It should call the mortality model and medication model first, then use the LLM only to explain the outputs safely

In [5]:
#==========================================================================
# SECTION 1: INSTALL THE REQUIRED LIBRARIES
#==========================================================================
!pip install numpy scipy scikit-learn
!pip install scikit-survival
!pip install -q openai pandas numpy scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [16]:
#==========================================================================
# SECTION 2: IMPORT THE REQUIRED LIBRARIES
#==========================================================================
import os
import json
import joblib
import pandas as pd
import numpy as np

from openai import OpenAI
from google.colab import files

In [17]:
#==========================================================================
# SECTION 3: DATA LOAD
#
# 1. Since the data set and pre-trained models are very large, they are saved on the
# google drive.
# 2. Shared path (View Access Only)
# https://drive.google.com/drive/folders/1RZD0xXibDHFEMcNWTQrqSVq-1HxX87JL?usp=sharing
# The drive path is shared in the readme file in Github repo as well
# Download the Data_Files and Models folder to your local drive.
# 3. Create the following folder structure in your google drive
# My Drive/ICU_Chatbot/Data_Files. Upload the csv file "female_icu_blood_disease_with_medications.csv", "female_blood_disease_icu.csv"
# to the location.
# My Drive/ICU_Chatbot/Models/. Upload
# "female_blood_patients_meds_Recmd_model.pkl" and "Female_Blood_Patients_Mortality_Pred_Model.pkl"
#==========================================================================

# Mount the google drive and authenticate once

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# ============================================================
# SECTION 4: Load dataset + BOTH models
# ============================================================

# Define file paths
# Update these paths based on your Drive structure (if needed)

# Example paths (edit if needed)
MORTALITY_DATA_PATH = "/content/drive/MyDrive/ICU_Chatbot/Data_Files/female_blood_disease_icu.csv"
MEDICATIONS_DATA_PATH = "/content/drive/MyDrive/ICU_Chatbot/Data_Files/female_icu_blood_disease_with_medications.csv"
MORTALITY_MODEL_PATH = "/content/drive/MyDrive/ICU_Chatbot/Models/Female_Blood_Patients_Mortality_Pred_Model.pkl"
MEDICATIONS_MODEL_PATH = "/content/drive/MyDrive/ICU_Chatbot/Models/female_blood_patients_meds_Recmd_model.pkl"

# Load mortality dataset
mortality_df = pd.read_csv(MORTALITY_DATA_PATH)

# Load medications dataset
medications_df = pd.read_csv(MEDICATIONS_DATA_PATH)

# Load models
mortality_model = joblib.load(MORTALITY_MODEL_PATH)
medications_model = joblib.load(MEDICATIONS_MODEL_PATH)

print("Mortality dataset:", mortality_df.shape)
print("Medications dataset:", medications_df.shape)
print("Mortality model:", type(mortality_model))
print("Medication model:", type(medications_model))

Mortality dataset: (9747, 14)
Medications dataset: (633247, 15)
Mortality model: <class 'sksurv.ensemble.forest.RandomSurvivalForest'>
Medication model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [19]:
# ============================================================
# SECTION 5: CHECK MODELS
# ============================================================

print(type(mortality_model))

# Check if model supports probabilities
if hasattr(mortality_model, "predict_proba"):
    print("mortality_model supports probability predictions ✅")
else:
    print("mortality_model does NOT support predict_proba ❌")

print(type(medications_model))

# Check if model supports probabilities
if hasattr(medications_model, "predict_proba"):
    print("medications_model supports probability predictions ✅")
else:
    print("medications_model does NOT support predict_proba ❌")



<class 'sksurv.ensemble.forest.RandomSurvivalForest'>
mortality_model does NOT support predict_proba ❌
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
medications_model supports probability predictions ✅


In [41]:
# ============================================================
# SECTION 6: MORTALITY PREDICTION CUSTOM ACTION
# NOTE: I have used RandomsurvivalForest model.This model
# is not a normal classifier and does not directly support prediction.
# Instead , it predicts risk scores over time based on features. This is what
# is used below to arrive at mortality prediction.
# ============================================================
import pandas as pd
import numpy as np

def mortality_prediction_action(patient_row, model, time_horizon=30):
    """
    Predict mortality risk using Random Survival Forest.

    Automatically computes duration_hours if missing using:
    - intime
    - outtime
    - deathtime
    """

    # -------------------------------
    # Step 1: Compute duration_hours
    # -------------------------------
    if "duration_hours" in patient_row.index:
        duration_hours = patient_row["duration_hours"]
    else:
        intime = pd.to_datetime(patient_row.get("intime"))
        outtime = pd.to_datetime(patient_row.get("outtime"))
        deathtime = pd.to_datetime(patient_row.get("deathtime"))

        if pd.notnull(deathtime):
            duration_hours = (deathtime - intime).total_seconds() / 3600
        else:
            duration_hours = (outtime - intime).total_seconds() / 3600

    # Safety check
    if duration_hours is None or duration_hours <= 0:
        duration_hours = 1.0  # fallback to avoid model crash

    # -------------------------------
    # Step 2: Build feature vector
    # -------------------------------
    feature_order = [
        "heart_rate_mean",
        "heart_rate_min",
        "heart_rate_max",
        "hemoglobin",
        "hematocrit",
        "glucose",
        "duration_hours"
    ]

    X = pd.DataFrame([{
        "heart_rate_mean": patient_row.get("heart_rate_mean", 0),
        "heart_rate_min": patient_row.get("heart_rate_min", 0),
        "heart_rate_max": patient_row.get("heart_rate_max", 0),
        "hemoglobin": patient_row.get("hemoglobin", 0),
        "hematocrit": patient_row.get("hematocrit", 0),
        "glucose": patient_row.get("glucose", 0),
        "duration_hours": duration_hours
    }])[feature_order]

    # -------------------------------
    # Step 3: Survival prediction
    # -------------------------------
    survival_function = model.predict_survival_function(X)[0]

    survival_probability = float(survival_function(time_horizon))
    mortality_probability = 1.0 - survival_probability

    # -------------------------------
    # Step 4: Risk stratification
    # -------------------------------
    if mortality_probability >= 0.70:
        risk = "High Risk"
    elif mortality_probability >= 0.40:
        risk = "Moderate Risk"
    else:
        risk = "Low Risk"

    return {
        "time_horizon": time_horizon,
        "duration_hours_used": round(duration_hours, 2),
        "survival_probability": round(survival_probability, 4),
        "mortality_probability": round(mortality_probability, 4),
        "risk_level": risk
    }


In [42]:
# ============================================================
# SECTION 7: MEDICATION PREDICTION CUSTOM ACTION
# ============================================================
def medication_recommendation_action(
    patient_medications,
    model,
    candidate_medications=None,
    top_n=10
):
    """
    Recommend medications based on predicted mortality risk reduction.

    This version assumes:
    - The trained model is a RandomForestClassifier
    - The target was mortality
    - The model uses ONLY medication columns as features
    - No clinical features are required

    Inputs:
    - patient_medications: list of medications the patient already has
      Example: ["heparin", "insulin", "vancomycin"]

    - model: trained RandomForestClassifier

    - candidate_medications: optional list of medication feature columns.
      If None, function uses model.feature_names_in_.

    Returns:
    - Ranked medication recommendations
    """

    import pandas as pd
    import numpy as np

    # --------------------------------------------
    # 1. Get medication feature names
    # --------------------------------------------
    if candidate_medications is None:
        if hasattr(model, "feature_names_in_"):
            candidate_medications = list(model.feature_names_in_)
        else:
            raise ValueError(
                "candidate_medications must be provided because model.feature_names_in_ is unavailable."
            )

    # --------------------------------------------
    # 2. Normalize patient medication names
    # --------------------------------------------
    patient_medications = [
        str(med).lower().strip()
        for med in patient_medications
    ]

    # --------------------------------------------
    # 3. Create current patient medication vector
    # --------------------------------------------
    X_patient = pd.DataFrame(
        np.zeros((1, len(candidate_medications))),
        columns=candidate_medications
    )

    for med in patient_medications:
        possible_names = [
            med,
            f"med_{med}",
            f"med_{med.replace(' ', '_')}"
        ]

        for name in possible_names:
            if name in X_patient.columns:
                X_patient.loc[0, name] = 1

    # --------------------------------------------
    # 4. Predict current mortality risk
    # --------------------------------------------
    current_risk = model.predict_proba(X_patient)[0][1]

    recommendations = []

    # --------------------------------------------
    # 5. Simulate adding each medication
    # --------------------------------------------
    for med_col in candidate_medications:

        # Skip medications already present
        if X_patient.loc[0, med_col] == 1:
            continue

        simulated_patient = X_patient.copy()
        simulated_patient.loc[0, med_col] = 1

        new_risk = model.predict_proba(simulated_patient)[0][1]
        risk_reduction = current_risk - new_risk

        recommendations.append({
            "medication": med_col.replace("med_", "").replace("_", " "),
            "current_mortality_risk": round(float(current_risk), 4),
            "predicted_risk_if_added": round(float(new_risk), 4),
            "risk_reduction": round(float(risk_reduction), 4)
        })

    rec_df = pd.DataFrame(recommendations)

    rec_df = rec_df.sort_values(
        by="risk_reduction",
        ascending=False
    )

    return rec_df.head(top_n).to_dict(orient="records")

In [ ]:
# ============================================================
# SECTION 8: ICU CHATBOT FUNCTION
# ============================================================

SYSTEM_PROMPT = """
You are an ICU clinical decision-support assistant.

You help health professionals interpret:
1. Mortality prediction model output
2. Medication recommendation model output

Rules:
- Do NOT prescribe medications.
- Do NOT claim causal benefit.
- Explain that recommendations are based on historical MIMIC-III model patterns.
- Always recommend clinician review.
- Keep answers clinically structured and concise.
"""

# Set your OpenAI API key and uncomment the line below before running this cell 

# os.environ["OPENAI_API_KEY"] = "REPLACE WITH YOUR OpenAI API KEY"

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


def icu_chatbot(
    question,
    mortality_patient_row,
    patient_medications,
    mortality_model,
    medications_model,
    top_n_meds=10
):
    """
    ICU chatbot that:
    1. Calls mortality_prediction_action()
    2. Calls medication_recommendation_action()
    3. Sends both model outputs to the LLM for safe clinical explanation
    """

    mortality_result = mortality_prediction_action(
        patient_row=mortality_patient_row,
        model=mortality_model
    )

    medication_result = medication_recommendation_action(
        patient_medications=patient_medications,
        model=medications_model,
        top_n=top_n_meds
    )

    context = {
        "patient_features_for_mortality_model": {
            "heart_rate_mean": float(mortality_patient_row["heart_rate_mean"]),
            "heart_rate_min": float(mortality_patient_row["heart_rate_min"]),
            "heart_rate_max": float(mortality_patient_row["heart_rate_max"]),
            "hemoglobin": float(mortality_patient_row["hemoglobin"]),
            "hematocrit": float(mortality_patient_row["hematocrit"]),
            "glucose": float(mortality_patient_row["glucose"])
        },
        "current_patient_medications": patient_medications,
        "mortality_prediction": mortality_result,
        "medication_recommendation": medication_result,
        "user_question": question
    }

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "Use the following ICU model outputs to answer safely.\n\n"
                    + json.dumps(context, indent=2)
                )
            }
        ]
    )

    return response.choices[0].message.content

In [44]:
# ============================================================
# SECTION 9: PREPARE CSV DATA FOR TEST CASES
# ============================================================

# Standardize column names
mortality_df.columns = (
    mortality_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

medications_df.columns = (
    medications_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

MORTALITY_FEATURES = [
    "heart_rate_mean",
    "heart_rate_min",
    "heart_rate_max",
    "hemoglobin",
    "hematocrit",
    "glucose"
]

missing_cols = [
    col for col in MORTALITY_FEATURES
    if col not in mortality_df.columns
]

if missing_cols:
    raise KeyError(
        f"Missing mortality feature columns: {missing_cols}\n"
        f"Available columns: {mortality_df.columns.tolist()}"
    )

mortality_test_df = mortality_df.dropna(subset=MORTALITY_FEATURES).copy()

print("Mortality test rows:", mortality_test_df.shape)
print("Medication rows:", medications_df.shape)

Mortality test rows: (1224, 14)
Medication rows: (633247, 15)


In [45]:
# ============================================================
# SECTION 10: HELPER FUNCTION TO GET MEDS FROM MEDICATION CSV
# ============================================================

def get_patient_medications_from_csv(
    medications_df,
    hadm_id=None,
    subject_id=None
):
    """
    Extract medication list from medications_df.
    Prefer hadm_id if available.
    """

    if hadm_id is not None and "hadm_id" in medications_df.columns:
        med_rows = medications_df[medications_df["hadm_id"] == hadm_id]
    elif subject_id is not None and "subject_id" in medications_df.columns:
        med_rows = medications_df[medications_df["subject_id"] == subject_id]
    else:
        raise ValueError("Need hadm_id or subject_id to find medications.")

    if "drug" not in med_rows.columns:
        raise KeyError(
            f"'drug' column not found. Available columns: {med_rows.columns.tolist()}"
        )

    meds = (
        med_rows["drug"]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .unique()
        .tolist()
    )

    return meds

In [46]:
# ============================================================
# SECTION 11: TEST CASE 1 — REAL PATIENT FROM CSV FILES
# ============================================================

patient_1 = mortality_test_df.iloc[0]

hadm_id_1 = patient_1["hadm_id"] if "hadm_id" in patient_1.index else None
subject_id_1 = patient_1["subject_id"] if "subject_id" in patient_1.index else None

patient_1_meds = get_patient_medications_from_csv(
    medications_df=medications_df,
    hadm_id=hadm_id_1,
    subject_id=subject_id_1
)

question_1 = """
Assess this ICU patient using the mortality model and medication recommendation model.
What is the mortality risk and which medications should the ICU team review?
"""

answer_1 = icu_chatbot(
    question=question_1,
    mortality_patient_row=patient_1,
    patient_medications=patient_1_meds,
    mortality_model=mortality_model,
    medications_model=medications_model,
    top_n_meds=10
)

print("========== TEST CASE 1: REAL CSV PATIENT ==========")
print("HADM_ID:", hadm_id_1)
print("SUBJECT_ID:", subject_id_1)
print("Current medications:", patient_1_meds[:20])
print("\nChatbot response:\n")
print(answer_1)

========== TEST CASE 1: REAL CSV PATIENT ==========
HADM_ID: 153559
SUBJECT_ID: 20542
Current medications: ['acetaminophen', 'acetylcysteine 20%', 'albuterol 0.083% neb soln', 'amp', 'atorvastatin', 'atropine sulfate', 'aztreonam', 'calcitriol', 'calcium acetate', 'calcium carbonate', 'calcium gluconate', 'ciprofloxacin hcl', 'd5w', 'diltiazem', 'diltiazem extended-release', 'diphenhydramine hcl', 'docusate sodium', 'dopamine', 'fentanyl citrate', 'ferrous sulfate']

Chatbot response:

### Mortality Risk Assessment

- **Survival Probability**: 60.32%
- **Mortality Probability**: 39.68%
- **Risk Level**: Low Risk
- **Time Horizon**: 30 days
- **Duration of Monitoring**: 22.62 hours

### Medication Recommendations for Review

The following medications are suggested based on historical patterns from the MIMIC-III model. Each medication is associated with a current mortality risk and a predicted risk if added:

1. **Lisinopril**
   - Current Mortality Risk: 54.17%
   - Predicted Risk if Ad

In [47]:
# ============================================================
# SECTION 12: TEST CASE 2 — ANOTHER REAL PATIENT FROM CSV FILES
# ============================================================

patient_2 = mortality_test_df.iloc[1]

hadm_id_2 = patient_2["hadm_id"] if "hadm_id" in patient_2.index else None
subject_id_2 = patient_2["subject_id"] if "subject_id" in patient_2.index else None

patient_2_meds = get_patient_medications_from_csv(
    medications_df=medications_df,
    hadm_id=hadm_id_2,
    subject_id=subject_id_2
)

question_2 = """
Review this second ICU patient.
Explain mortality risk level and medication recommendations.
"""

answer_2 = icu_chatbot(
    question=question_2,
    mortality_patient_row=patient_2,
    patient_medications=patient_2_meds,
    mortality_model=mortality_model,
    medications_model=medications_model,
    top_n_meds=10
)

print("========== TEST CASE 2: SECOND REAL CSV PATIENT ==========")
print("HADM_ID:", hadm_id_2)
print("SUBJECT_ID:", subject_id_2)
print("Current medications:", patient_2_meds[:20])
print("\nChatbot response:\n")
print(answer_2)

========== TEST CASE 2: SECOND REAL CSV PATIENT ==========
HADM_ID: 186403
SUBJECT_ID: 2066
Current medications: ['1/2 ns', 'acetaminophen', 'acetaminophen w/codeine', 'albumin 5% (25 g)', 'albumin, human', 'albuterol', 'albuterol 0.083% neb soln', 'albuterol-ipratropium', 'alteplase (catheter clearance)', 'amino acids 4.25% w/ dextrose 5%', 'amiodarone hcl', 'artificial tears', 'aspirin', 'aspirin ec', 'atorvastatin', 'bacitracin/polymyxin b sulfate opht. oint', 'bag', 'bisacodyl', 'brimonidine tartrate 0.15% ophth.', 'calcium gluconate']

Chatbot response:

### Mortality Risk Level

- **Survival Probability**: 98.87%
- **Mortality Probability**: 1.13%
- **Risk Level**: Low Risk

The patient exhibits a low risk of mortality over the next 30 days based on the model's output, which suggests a favorable prognosis. The model utilized historical data patterns from the MIMIC-III database to arrive at this assessment. Clinician review is recommended to consider the patient's overall clinical

In [48]:
# ============================================================
# SECTION 13: CLINICAL SAFETY NOTE
# ============================================================

print("""
Clinical Safety Note:
This ICU chatbot is a decision-support prototype only.

The mortality model and medication recommendation model are based on
retrospective MIMIC-III data. Medication outputs should be interpreted as
items for clinician review, not treatment orders.

The system should not be used as a substitute for physician judgment,
hospital protocols, medication contraindication checks, allergy checks,
renal/hepatic dosing review, or bedside clinical assessment.
""")


Clinical Safety Note:
This ICU chatbot is a decision-support prototype only.

The mortality model and medication recommendation model are based on
retrospective MIMIC-III data. Medication outputs should be interpreted as
items for clinician review, not treatment orders.

The system should not be used as a substitute for physician judgment,
hospital protocols, medication contraindication checks, allergy checks,
renal/hepatic dosing review, or bedside clinical assessment.

